# Probability and Information Theory: A Mathematical Map for LLM Engineers

> Across the previous Parts a few terms kept reappearing: softmax, cross-entropy, KL, perplexity, temperature. Each section used them from an engineering angle, but nowhere did we lay out their mathematical relationships clearly. This section assembles these concepts into a single map.
>
> In this section, we start from the categorical distribution, walk the line "sampling -> information theory -> CE loss -> gradient -> reuse", and end with a reuse map that aligns training, RLHF, DPO, distillation, and MoE within the same mathematical structure.

What an LLM does at each position can be summarized in one sentence: it outputs a categorical distribution describing "the probability that the next token is each candidate in the vocabulary". This sentence looks plain, but it ties together four stages - training, inference, evaluation, and alignment - because every stage operates on the same distribution, just from a different viewpoint.

During training, we want this distribution to be as close as possible to the one-hot distribution of the true answer, with the distance measured by cross-entropy or KL; during inference, we sample from the distribution, and temperature, top-k, and top-p are all modifications to the shape of the distribution; during evaluation, perplexity is an exponentiated wrapper around cross-entropy; during alignment, RLHF uses a KL penalty to prevent policy drift, and DPO uses a log-ratio to indirectly measure distributional shift. Understanding the categorical distribution and the three information-theoretic quantities around it (entropy, cross-entropy, KL) is equivalent to acquiring the universal vocabulary of LLM engineering.

This section assumes the reader has finished 11-training-loss and 19-generation. The former covered how CE loss is computed, the latter covered the engineering usage of temperature/top-k/top-p. What we fill in here is the mathematical skeleton behind them, and at the end we map these concepts back to sub-fields like RLHF, DPO, distillation, and MoE, so that from now on no terminology will trip you up when reading any LLM paper.


## 1. The Categorical Distribution: The Essence of LLM Output

Think of a vocabulary with V tokens as V mutually exclusive categories. What the model produces at each position is not a single token, but a probability assignment over all categories $p = (p_1, \ldots, p_V)$, satisfying $p_i \ge 0$ and $\sum_i p_i = 1$. A distribution that "assigns probability over a finite set of categories" is called a categorical distribution.

As an example, suppose V=4 and the model produces logits (defined below) $z = [2.0, 1.0, 0.1, -1.0]$ at some position. These logits are not yet probabilities, but they uniquely determine a categorical distribution. The conversion from logits to probabilities is performed by softmax.

Expert routing in MoE is another typical application of the categorical distribution: the router outputs num_experts scores per token, and after softmax these scores become "the probability that this token is assigned to each expert". So conceptually, next-token prediction and expert routing are the same thing - both assign probability over a finite set of categories.


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Use a tiny set of numbers to demonstrate the categorical distribution
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
print("logits:           ", logits.tolist())
print("exp(logits):      ", [round(x, 4) for x in torch.exp(logits).tolist()])
print("normalized probs: ", [round(x, 4) for x in F.softmax(logits, dim=-1).tolist()])
print("sum of probs:     ", round(F.softmax(logits, dim=-1).sum().item(), 6))
print()
print("Key observation: logits are arbitrary real numbers; softmax compresses them into a set of non-negative probabilities that sum to 1.")
print("This is exactly the parameterized form of a categorical distribution: V categories, one probability per category.")


### 1.1 From Logits to Probabilities: The Role of Softmax

Softmax maps any real vector $z \in \mathbb{R}^V$ onto the probability simplex:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

The exponential $e^{z_i}$ is always positive, so the numerator guarantees non-negativity; the denominator sums all terms to guarantee normalization. The relative ordering is preserved: $z_i > z_j \Rightarrow p_i > p_j$. So argmax over the logits and over the softmax output land on the same position - greedy decoding can therefore directly take the argmax of the logits.

Softmax is not the only function that turns logits into probabilities (for instance `sigmoid + normalization` also works), but it is the only differentiable choice for which "maximum likelihood is equivalent to minimizing cross-entropy". This equivalence is the mathematical root of why LM training can use CE loss.


### 1.2 Numerically Stable Version: logsumexp

Computing softmax directly from the definition is numerically unsafe. When $z_i$ is large, $e^{z_i}$ overflows to `inf`. The fix relies on softmax being invariant under a constant shift:

$$\text{softmax}(z)_i = \frac{e^{z_i - c}}{\sum_j e^{z_j - c}}$$

where $c$ is usually taken as $c = \max_j z_j$. This way the largest exponential becomes $e^0 = 1$, and all the rest are less than 1, so nothing overflows.

A further refinement is logsumexp: when computing $\log \sum_j e^{z_j}$ directly, the same shift is applied, yielding

$$\text{logsumexp}(z) = c + \log \sum_j e^{z_j - c}$$

Thus the numerically stable log-softmax can be written as $\log p_i = z_i - \text{logsumexp}(z)$. This identity is the starting point of FlashAttention's online softmax - when V is large and cannot fit in memory at once, logsumexp can be accumulated in a chunked, streaming fashion.

PyTorch's `F.log_softmax` and `F.cross_entropy` both use logsumexp internally, so normally you do not have to write it yourself. But understanding this step is necessary for reading the kernel implementations of inference frameworks such as vLLM and TGI.


In [ ]:
# Compare: naive softmax vs numerically stable softmax
big_logits = torch.tensor([1000.0, 1001.0, 1002.0])

naive_exp = torch.exp(big_logits)
print("Naive implementation:")
print("  exp([1000, 1001, 1002]) =", naive_exp.tolist())
print("  -> all overflow to inf, cannot normalize")
print()

# Subtract the max first, then proceed
stable_shifted = big_logits - big_logits.max()
stable_exp = torch.exp(stable_shifted)
stable_probs = stable_exp / stable_exp.sum()
print("Numerically stable version (subtract max=1002):")
print("  shifted logits =", stable_shifted.tolist())
print("  exp(shifted)   =", [round(x, 6) for x in stable_exp.tolist()])
print("  softmax        =", [round(x, 4) for x in stable_probs.tolist()])
print()

# PyTorch does this internally
torch_logsoftmax = F.log_softmax(big_logits, dim=-1)
print("PyTorch log_softmax:", [round(x, 4) for x in torch_logsoftmax.tolist()])
print("Key observation: subtracting the max does not change the softmax result, but avoids exp overflow.")


## 2. A Probabilistic View of Sampling

Drawing a token from a categorical distribution during inference is, in probability theory, simply "sampling". The temperature, top-k, and top-p covered in 19-generation are essentially different forms of modification applied to the categorical distribution before sampling. Once we put them back into probabilistic language, the engineering parameters acquire precise mathematical meaning.

Below we use the same set of logits to show how the three sampling methods change the output distribution.


### 2.1 Temperature: Divide the Logits by T

The formula for temperature sampling is $p^{(T)} = \text{softmax}(z / T)$. Note that what gets divided is the logits, not the probabilities. This matters - temperature operates on "unnormalized scores" rather than directly adjusting $p_i$.

Two extremes:

- $T \to 0^+$: $z/T$ grows toward infinity, the largest term's exponential dominates the rest, and the softmax output degenerates to one-hot. This is equivalent to argmax, i.e. greedy decoding.
- $T \to \infty$: $z/T \to 0$, all $e^{z_i/T} \to 1$, and the softmax output tends toward the uniform distribution. This is equivalent to "guessing uniformly at random across all tokens".

When T=1 the distribution exactly matches the softmax of the original logits. So T is a continuous knob whose two ends correspond to "deterministic" and "uniform".


In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
print(f"{'T':>5}  {'p0':>7}  {'p1':>7}  {'p2':>7}  {'p3':>7}  note")
print("-" * 70)

for T in [0.1, 0.5, 1.0, 2.0, 10.0, 1000.0]:
    probs = F.softmax(logits / T, dim=-1)
    row = f"{T:>5.1f}  " + "  ".join(f"{p:>7.4f}" for p in probs.tolist())
    if T <= 0.1:
        note = "approaches argmax (greedy)"
    elif T == 1.0:
        note = "original distribution"
    elif T >= 1000.0:
        note = "approaches uniform"
    else:
        note = ""
    print(row + "  " + note)

print()
print("Key observation: T is a continuous knob. T->0 compresses the distribution into one-hot, T->inf stretches it into uniform.")


### 2.2 Top-k: Truncate Then Renormalize

The probabilistic meaning of top-k is: truncate the categorical distribution to the k categories with the highest probability, set the probability of the remaining categories to 0, and then renormalize the remaining k. This is a two-step operation of "first mask, then renormalize".

In formula form: let $S_k$ be the set of indices of the k categories with the highest probability, then

$$p^{(\text{top-}k)}_i = \begin{cases} \frac{p_i}{\sum_{j \in S_k} p_j} & i \in S_k \\ 0 & i \notin S_k \end{cases}$$

k=1 is a special case: truncate down to a single category, and after renormalization the result is necessarily one-hot. This is equivalent to greedy decoding.

In engineering implementations we usually do not explicitly set values to 0; instead we set the logits at the truncated positions to $-\infty$, and softmax automatically turns them into 0. This is the same trick as the `-inf` in masked attention.


### 2.3 Top-p (Nucleus): Dynamically Choose the Smallest Set

The k in top-k is fixed, but the shape of the probability distribution varies a lot across positions - when the distribution is sharply peaked, k=40 is too many; when it is nearly flat, k=40 is too few. Top-p instead truncates by cumulative probability:

Sort the categories by probability from largest to smallest, and denote the sorted probabilities as $p_{(1)} \ge p_{(2)} \ge \ldots$. Top-p selects the smallest prefix set $S_p$ such that $\sum_{i \in S_p} p_{(i)} \ge p$. Positions outside the set are likewise set to 0 and then renormalized.

When p=1 this is equivalent to no truncation at all (pure sampling); when p is very small it is equivalent to greedy. There is one detail in the HuggingFace implementation: the token whose cumulative probability "just exceeds p" is kept rather than discarded - so for p=0.9 the cumulative probability actually kept may be slightly larger than 0.9.

The differences in output among the three sampling methods on the same set of logits are directly visible in the code below.


In [ ]:
# Use a relatively peaked set of logits to show the output differences among the three sampling methods
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, 0.0, -1.0])
base_probs = F.softmax(logits, dim=-1)
print("original logits:", logits.tolist())
print("original probs: ", [round(p, 4) for p in base_probs.tolist()])
print()

# Temperature: T=0.5
probs_t = F.softmax(logits / 0.5, dim=-1)
print("Temperature=0.5 (low temperature, more peaked):")
print("  ", [round(p, 4) for p in probs_t.tolist()])
print()

# Top-k=2: keep the top 2
top2_vals, top2_idx = torch.topk(logits, 2)
mask_k = torch.full_like(logits, float('-inf'))
mask_k[top2_idx] = top2_vals
probs_k = F.softmax(mask_k, dim=-1)
print("Top-k=2 (truncate to 2):")
print("  ", [round(p, 4) if p > 0 else '  0   ' for p in probs_k.tolist()])
print()

# Top-p=0.8: cumulative probability reaches 0.8
sorted_probs, sorted_idx = torch.sort(base_probs, descending=True)
cum = torch.cumsum(sorted_probs, dim=0)
keep_mask = cum <= 0.8
keep_mask[0] = True  # keep at least one
# Also keep the token whose cumulative value first exceeds 0.8
first_exceed = (cum > 0.8).nonzero()
if len(first_exceed) > 0:
    keep_mask[first_exceed[0].item()] = True
kept_idx = sorted_idx[keep_mask]
mask_p = torch.full_like(logits, float('-inf'))
mask_p[kept_idx] = logits[kept_idx]
probs_p = F.softmax(mask_p, dim=-1)
print("Top-p=0.8 (cumulative probability truncation):")
print("  ", [round(p, 4) if p > 0 else '  0   ' for p in probs_p.tolist()])
print()
print("Key observations:")
print("  Temperature changes the relative ratio of all probabilities, but zeroes out no category")
print("  Top-k truncates by a fixed count, Top-p truncates by cumulative probability")
print("  The industrial default is to use Top-k together with Top-p: first fix an upper bound, then adapt")


### 2.4 Gumbel-Max Trick (Supplement)

There is another equivalent way to sample from a categorical distribution, called the Gumbel-Max trick: add an i.i.d. Gumbel noise $g_i \sim \text{Gumbel}(0, 1)$ to each logit and then take argmax. It can be shown that the argmax obtained this way follows the original categorical distribution.

This trick may look like a mathematical sleight of hand, but it moves "the randomness of sampling" from multinomial sampling to "add noise then take argmax" - and the latter is naturally differentiable (argmax approximated by softmax), so during training gradients can also flow through the sampling step. This is the reparameterization idea from RL. A detailed derivation is beyond the scope of this appendix; see the [cs231n note on Gumbel-Softmax](https://cs231n.github.io/) and the original paper [Gumbel-Softmax](https://arxiv.org/abs/1611.01144).


## 3. The Three Tools of Information Theory

Above we covered "what a distribution is" and "how to sample from it". Next we cover "how to measure the uncertainty of a distribution, and the difference between two distributions". These are the three core quantities of information theory: entropy, cross-entropy, and KL divergence. The relationship among them is the single most important identity in LLMs.

Below we use two concrete examples - a "fair die" and a "biased die" - to demonstrate, so we do not just pile on formulas.


### 3.1 Entropy: The Uncertainty of a Distribution

Entropy measures how "uncertain" a distribution $p$ is in itself. The formula:

$$H(p) = -\sum_x p(x) \log p(x)$$

Intuition: $-\log p(x)$ is "the amount of surprise on seeing $x$" - low-probability events are more surprising when they occur. Entropy is "the expected amount of surprise".

Two examples with a 6-sided die:

- Fair die: each face has probability 1/6, the next outcome is completely unpredictable, and entropy is maximal.
- Biased die: one face has 99%, the other faces together add up to 1%, the outcome is almost surely that face, and entropy is close to 0.

For a categorical distribution with V categories, the maximum entropy is $\log V$, attained under the uniform distribution; the minimum entropy is 0, attained under one-hot. If the entropy of an LLM's prediction distribution is high, the model is very unsure; if it is low, the model is very confident (but possibly confidently wrong).


In [ ]:
# Compare entropy using a 6-sided die
import math

fair_die = torch.tensor([1/6] * 6)
biased_die = torch.tensor([0.99, 0.002, 0.002, 0.002, 0.002, 0.002])
one_hot_die = torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])

def entropy(p, eps=1e-12):
    """Compute the entropy of a discrete distribution; add eps to avoid log(0)"""
    return -(p * torch.log(p + eps)).sum().item()

print(f"fair die entropy:    {entropy(fair_die):.4f}  (maximum log(6) = {math.log(6):.4f})")
print(f"biased die entropy:  {entropy(biased_die):.4f}  (almost certain, close to 0)")
print(f"one-hot entropy:     {entropy(one_hot_die):.4f}  (completely certain, equals 0)")
print()
print("Key observation: entropy is maximal for uniform and minimal (0) for one-hot.")
print("During LM training, the ground-truth one-hot distribution has entropy = 0; this will be used repeatedly later.")


### 3.2 Cross-Entropy: The Cost of Encoding p with q

Cross-entropy measures "the average number of bits needed when using distribution $q$ to encode events drawn from the true distribution $p$":

$$H(p, q) = -\sum_x p(x) \log q(x)$$

Note that the two distributions play different roles: $p$ is the "true" distribution and $q$ is the "predicted" distribution. $-\log q(x)$ is "the number of bits needed to encode $x$ under $q$", and the whole expression is a probability-weighted average over $p$.

If $q$ puts high probability on the high-probability events of $p$, cross-entropy is small; if $q$ assigns probability incorrectly, cross-entropy is large. In LM training $p$ is the ground truth (one-hot) and $q$ is the model's predicted distribution, so minimizing cross-entropy means making $q$ concentrate its probability on the correct token.


### 3.3 KL Divergence: The Difference Between Two Distributions

KL divergence measures the difference between two distributions:

$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)}$$

It has several important properties:

- Non-negative: $D_{KL}(p \| q) \ge 0$, with equality if and only if $p = q$.
- Asymmetric: $D_{KL}(p \| q) \ne D_{KL}(q \| p)$, so it is a "divergence", not a "distance".
- $D_{KL}(p \| q)$ contributes 0 wherever $p$ has probability 0 (under the convention $0 \log 0 = 0$), but if $q$ has probability 0 while $p$ does not, it diverges to infinity.

RLHF uses $D_{KL}(\pi_\theta \| \pi_{ref})$ to prevent the new policy from drifting too far from the reference policy; distillation uses $D_{KL}(p_{teacher} \| p_{student})$ to have the student imitate the teacher. The choice of KL direction in both cases is tied to "the support of the student / new policy must contain that of the teacher / reference policy" - choosing the wrong direction makes training diverge.


### 3.4 The Core Identity: H(p, q) = H(p) + D_KL(p || q)

This is the single most important information-theoretic identity in LLMs. The derivation is short - just expand the definitions:

$$D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)} = \sum_x p(x) \log p(x) - \sum_x p(x) \log q(x) = -H(p) + H(p, q)$$

Rearranging gives

$$H(p, q) = H(p) + D_{KL}(p \| q)$$

Meaning: the cost of encoding $p$ with $q$ = the inherent uncertainty of $p$ + the extent to which $q$ deviates from $p$. The former is incompressible; the latter is what the model can improve.

This identity explains why LM training uses cross-entropy rather than KL directly - the two differ by a constant $H(p)$ that has no effect on the gradient, but the cross-entropy form is simpler (it needs only $\log q$, not $\log p$). We verify this with the die example below.


In [ ]:
# Verify with dice: H(p, q) = H(p) + D_KL(p || q)
def cross_entropy(p, q, eps=1e-12):
    """Cost of encoding p with q"""
    return -(p * torch.log(q + eps)).sum().item()

def kl_divergence(p, q, eps=1e-12):
    """D_KL(p || q)"""
    return (p * (torch.log(p + eps) - torch.log(q + eps))).sum().item()

p = torch.tensor([0.4, 0.3, 0.2, 0.1])  # true distribution
q_good = torch.tensor([0.35, 0.35, 0.2, 0.1])  # prediction close to p
q_bad = torch.tensor([0.1, 0.2, 0.3, 0.4])     # prediction far from p

print("true distribution p:", p.tolist())
print()

for name, q in [("q_good (close to p)", q_good), ("q_bad (far from p)", q_bad)]:
    H_p = entropy(p)
    H_pq = cross_entropy(p, q)
    kl = kl_divergence(p, q)
    lhs = H_pq
    rhs = H_p + kl
    print(f"{name}:")
    print(f"  q             = {[round(x, 2) for x in q.tolist()]}")
    print(f"  H(p)          = {H_p:.4f}")
    print(f"  H(p, q)       = {H_pq:.4f}")
    print(f"  D_KL(p || q)  = {kl:.4f}")
    print(f"  H(p)+D_KL     = {rhs:.4f}  (should equal H(p,q) = {lhs:.4f})")
    print()

print("Key observation: H(p,q) and H(p)+D_KL are numerically identical.")
print("The closer q is to p, the smaller D_KL becomes; when q = p, D_KL = 0 and H(p,q) = H(p).")


## 4. Why LM Loss Uses CE

Hooking the identity from Section 3 up to LM training reveals a clean chain of equivalences: maximum likelihood -> minimize NLL -> minimize cross-entropy -> minimize KL (off by a constant). This section walks through the chain and explains why the true distribution under teacher forcing is one-hot.

**Maximum likelihood**: given training corpus $x_1, \ldots, x_T$, the goal is to maximize the probability the model assigns to this sequence $\prod_t p_\theta(x_t | x_{<t})$. This is equivalent to maximizing the log-likelihood $\sum_t \log p_\theta(x_t | x_{<t})$, and also equivalent to minimizing the negative log-likelihood (NLL):

$$\mathcal{L}_{\text{NLL}} = -\sum_t \log p_\theta(x_t | x_{<t})$$

Now we write NLL in the form of cross-entropy. At each position, the model outputs a distribution $q = p_\theta(\cdot | x_{<t})$. Teacher forcing assumes the true distribution $p$ is one-hot - "the next token is exactly the one in the training data", with no other possibility. So $p$ has probability 1 on the correct token $x_t$ and 0 elsewhere. Substituting into the cross-entropy definition:

$$H(p, q) = -\sum_x p(x) \log q(x) = -\log q(x_t) = -\log p_\theta(x_t | x_{<t})$$

So NLL = cross-entropy (under teacher forcing the two are exactly equal). Substituting further into the core identity $H(p, q) = H(p) + D_{KL}(p \| q)$: the $H(p)$ of a one-hot distribution is 0, so cross-entropy = KL divergence.

The three quantities are completely equal in LM training:

$$\mathcal{L}_{\text{NLL}} = H(p, q) = D_{KL}(p \| q)$$

This is why 11-training-loss uses `F.cross_entropy` directly - mathematically they are the same thing, just written differently.


In [ ]:
# Verify with tiny numbers: NLL = H(p, q) = D_KL(p || q) when p is one-hot
logits = torch.tensor([[2.0, 1.0, 0.5, -0.5]])  # one position, 4 classes
target_id = 1  # correct token

# Method 1: PyTorch's cross_entropy (internally this is NLL)
ce_loss = F.cross_entropy(logits, torch.tensor([target_id])).item()

# Method 2: compute NLL by hand = -log p(correct)
log_probs = F.log_softmax(logits, dim=-1)
nll = -log_probs[0, target_id].item()

# Method 3: build one-hot p and compute KL
p_onehot = torch.zeros(4)
p_onehot[target_id] = 1.0
q = F.softmax(logits, dim=-1)[0]
# KL(p || q) = sum p log(p/q), p is non-zero only at target_id, so this = log(1/q[target]) = -log q[target]
kl = (p_onehot * (torch.log(p_onehot + 1e-12) - torch.log(q + 1e-12))).sum().item()

print(f"PyTorch cross_entropy: {ce_loss:.6f}")
print(f"manual NLL:            {nll:.6f}")
print(f"KL(p || q):            {kl:.6f}")
print()
print("All three numbers are exactly equal - verifying that under LM training, NLL = CE = KL.")
print("Key observation: H(p) = 0 because p is one-hot, so there is no constant gap between CE and KL.")


## 5. The Engineering Meaning of Perplexity

Perplexity (PPL) is one of the most common metrics for evaluating LMs. Its definition is very simple:

$$\text{PPL} = \exp(\text{CE})$$

where CE is the average cross-entropy of the model on the test set (using the natural logarithm as the base). If log2 is the base, PPL = $2^{H(p, q)}$.

Why call it "perplexity"? You can think of it this way: CE is the number of bits of "average surprise" the model experiences at each position. After exponentiating, PPL represents "how many candidates the model is effectively choosing among uniformly at each position". For example:

- PPL = 1: the model is 100% certain of the correct token, with no hesitation (perfect).
- PPL = V (vocabulary size): the model is guessing uniformly across all V tokens (uniform).
- PPL = 10: the model is effectively "hesitating uniformly among 10 candidates".

The lower the PPL, the more certain the model is; but "certain" does not mean "correct" - it just means the model is confident in its guess.

**Not comparable across datasets**: PPL depends heavily on tokenization. The same model might have a PPL of 5 on a dataset with a BPE vocabulary of size 50000, and a PPL of 3 on a dataset with a unigram vocabulary of size 100000. A smaller number does not mean a better model - it may just be that the vocabulary is larger, so each token has a "coarser granularity" and is naturally easier to predict. So PPL can only be used to compare models on "the same dataset + the same tokenizer", not across datasets.


In [ ]:
# Demonstrate PPL computation on a simple sequence
torch.manual_seed(0)

# Suppose vocabulary V=10, and the model produces these logits at 4 positions
V = 10
logits = torch.tensor([
    [3.0, 1.0, 0.5, -0.5, 0.0, 0.2, -1.0, 0.1, -0.3, 0.4],  # position 0
    [0.1, 4.0, 0.2, -1.0, 0.0, 0.1, 0.0, -0.5, 0.3, -0.2],  # position 1 (very confident)
    [0.3, 0.2, 0.4, 0.1, 0.5, 0.2, 0.3, 0.4, 0.1, 0.5],     # position 2 (very unsure)
    [1.5, 0.5, 2.0, -0.5, 0.0, 0.3, -1.0, 0.2, -0.3, 0.1],  # position 3
])
targets = torch.tensor([0, 1, 4, 2])

# Compute CE
ce = F.cross_entropy(logits, targets).item()
ppl = math.exp(ce)

print(f"vocabulary size V = {V}")
print(f"sequence length T = {len(targets)}")
print(f"average CE   = {ce:.4f}")
print(f"PPL          = exp(CE) = {ppl:.4f}")
print()
print(f"reference values:")
print(f"  PPL=1 means perfect prediction, PPL={V} means random guessing")
print(f"  current PPL={ppl:.2f}, somewhere in between")
print(f"  position 2 has nearly uniform logits and contributes the most to CE; position 1 is confident and contributes the least")
print()

# Per-position CE
per_pos_ce = F.cross_entropy(logits, targets, reduction='none')
print("per-position CE: ", [round(x, 3) for x in per_pos_ce.tolist()])
print("per-position PPL:", [round(math.exp(x), 2) for x in per_pos_ce.tolist()])
print()
print("Key observation: PPL = exp(CE), an exponentiated wrapper around CE.")
print("PPL is not comparable across datasets - different tokenizers mean different scales of CE.")


## 6. An Engineering View of the Softmax + CE Gradient

This section covers a counterintuitive but extremely important result: the gradient of the softmax + cross-entropy combination has an extremely clean form -

$$\frac{\partial L}{\partial z} = p - \text{one\_hot}(t)$$

where $z$ is the logits, $p$ is the softmax output, and $t$ is the correct class. That is: the gradient at the correct position is $p_t - 1$ (negative, pushing down), and the gradient at the other positions is $p_i$ (positive, pulling down - but since the target $p_i$ should be small, the direction of the pull is correct).

Intuitive derivation (not rigorous; for the rigorous version see the [cs231n optimization note](https://cs231n.github.io/optimization-2/)):

1. Loss $L = -\log p_t$, so differentiating with respect to $p_t$ gives $\partial L / \partial p_t = -1/p_t$.
2. The Jacobian of softmax satisfies $\partial p_j / \partial z_i = p_j(\delta_{ij} - p_i)$, where $\delta_{ij}$ is the Kronecker delta.
3. Chain rule: $\partial L / \partial z_i = \sum_j (\partial L / \partial p_j)(\partial p_j / \partial z_i)$, only the $j = t$ term is non-zero.
4. Substituting and simplifying: $\partial L / \partial z_i = -1/p_t \cdot p_t(\delta_{it} - p_i) = p_i - \delta_{it}$.

Written as a vector this becomes $\partial L / \partial z = p - \text{one\_hot}(t)$.

Why does this gradient form make training stable? Because it is "bounded" - $p_i \in [0, 1]$, so every component of the gradient lies in $[-1, 1]$, and unlike sigmoid + MSE it will neither vanish nor explode. This is also why every classification task uses softmax + CE rather than other combinations.

PyTorch's `F.cross_entropy` internally fuses softmax and CE into a single kernel (called `log_softmax + NLL`); it does not explicitly construct $p$, but directly computes $z - \text{logsumexp}(z)$ and then takes the correct position - numerically stable, with the correct gradient.


In [ ]:
# Verify the softmax + CE gradient formula: dL/dz = p - one_hot(t)
logits = torch.tensor([2.0, 1.0, 0.5, -0.5], requires_grad=True)
target_id = 1

# Method 1: PyTorch autograd
loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([target_id]))
loss.backward()
auto_grad = logits.grad.clone()

# Method 2: manually compute p - one_hot(t)
with torch.no_grad():
    p = F.softmax(logits, dim=-1)
    one_hot = torch.zeros(4)
    one_hot[target_id] = 1.0
    manual_grad = p - one_hot

print(f"target_id = {target_id}")
print(f"softmax output p:  {[round(x, 4) for x in p.tolist()]}")
print(f"one_hot(t):        {one_hot.tolist()}")
print()
print(f"PyTorch autograd:  {[round(x, 4) for x in auto_grad.tolist()]}")
print(f"manual p-one_hot:  {[round(x, 4) for x in manual_grad.tolist()]}")
print()
print("The two are exactly equal, verifying dL/dz = p - one_hot(t).")
print()
print("Interpreting the signs of the gradient components:")
for i in range(4):
    g = manual_grad[i].item()
    if i == target_id:
        print(f"  position {i} (correct): gradient {g:+.4f} -> negative, pushing down (decreasing z is equivalent to increasing p)")
    else:
        print(f"  position {i} (wrong):   gradient {g:+.4f} -> positive, pulling down (decreasing z is equivalent to decreasing p)")
print()
print("Key observation: all gradient components lie in [-1, 1]; this boundedness is what makes softmax+CE training stable.")


## 7. The Reuse Map of Probability and Information Theory in LLM Engineering

The previous six sections built up the vocabulary. This section maps these concepts back to the various sub-fields of LLM engineering, so you can see that training, RLHF, DPO, distillation, evaluation, inference, and MoE all operate on the same set of mathematical objects.

This table is the core value of this appendix - after reading it, you should be able to find the corresponding mathematical structure in the table for any LLM paper you read.

| Sub-field | Core concepts used | Mathematical form | Intuition |
|:---|:---|:---|:---|
| Pretraining / SFT loss | cross-entropy (= NLL = KL) | $-\log p_\theta(x_t \| x_{<t})$ | maximize probability of the correct token |
| Evaluation | perplexity | $\exp(\text{CE})$ | effective number of candidates per position |
| Inference: temperature | softmax temperature | $\text{softmax}(z/T)$ | T->0 greedy, T->inf uniform |
| Inference: top-k / top-p | categorical truncation + renormalization | $p^{(S)} / \sum_{j \in S} p_j$ | restrict the candidate set |
| RLHF (PPO) | KL penalty | $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$ | prevent policy drift and reward hacking |
| DPO | log-ratio (implicit KL) | $\beta \log \frac{\pi_\theta}{\pi_{ref}}$ | turn preference into classification |
| Distillation | KL (forward or reverse) | $D_{KL}(p_{teacher} \| p_{student})$ | student imitates teacher's distribution |
| MoE routing | categorical distribution | $\text{softmax}(W_{gate} x)$ | assign each token a probability over experts |
| MoE load balancing | entropy / KL auxiliary loss | $\alpha \cdot \text{aux}(\bar{p})$ | prevent uneven expert assignment |

A few easily confused points are worth pulling out separately:

- **The KL direction in RLHF is $D_{KL}(\pi_\theta \| \pi_{ref})$** (forward KL), not reverse. The reason is that forward KL does not blow up to infinity where $\pi_{ref}$ has probability 0 (because the $\pi_\theta$ probability is small and the product goes to 0), so the new policy is allowed to explore situations the reference policy has never seen; if reverse KL were used instead, the new policy would be strictly confined to the support of $\pi_{ref}$ and could not explore.
- **The KL direction in distillation is usually $D_{KL}(p_{teacher} \| p_{student})$** (forward), so the student covers every position the teacher could output. There are also reverse-KL variants whose behavior differs.
- **DPO has no explicit KL, but the log-ratio $\log(\pi_\theta / \pi_{ref})$ is an implicit KL term**. It measures "the change in probability that the new policy assigns to a given answer relative to the reference policy", equivalent to a KL-constrained optimization in preference space.
- **The form of MoE load balancing loss varies**: some use entropy (encouraging a uniform router distribution), some use KL (aligning the router distribution with uniform), some use "the variance of expert utilization". But underneath they are all shape constraints on a categorical distribution.

Memorize this table; from now on, when reading any LLM paper, first ask: "Which distribution is it operating on? Which divergence is it minimizing?" - the vast majority of papers can be summarized by these two questions.


## 8. A Probabilistic View of the MoE Scenario

MoE (Mixture of Experts) is the scenario in LLM engineering where the categorical distribution and KL divergence are most intensively applied. This section expands the two MoE-related rows of the table in Section 7.

10-moe covered the structure of the MoE layer: each token passes through a router (a linear layer) that outputs num_experts scores; the top-k experts with the highest scores are taken, and their outputs are mixed by softmax weights. From a probabilistic viewpoint, the router output is exactly a categorical distribution - "the probability that this token is assigned to each expert".

**The router's softmax is a categorical distribution**. This is completely isomorphic to next-token prediction: the LM head outputs a categorical distribution over the vocabulary, and the router outputs a categorical distribution over the expert set. The mathematical structure is identical; only the meaning of the categories differs.

**The probabilistic form of the load balancing loss**. If the router is trained freely, it is easy to end up in a situation where "a few experts are chosen heavily while others sit idle" - this degenerates the MoE into a dense model and loses the advantage of sparsity. The fix is to add an auxiliary loss that penalizes uneven expert assignment.

Probabilistic readings of several common forms:

1. **Entropy form**: $L_{aux} = -H(\bar{p})$, where $\bar{p}$ is the average routing distribution over all tokens in the batch. Higher entropy means a more uniform distribution, so minimizing $-H$ encourages uniform assignment. This is equivalent to "making the per-batch average probability of each expert being chosen close to 1/num_experts".

2. **KL form**: $L_{aux} = D_{KL}(\bar{p} \| u)$, where $u$ is the uniform distribution. This is more direct than the entropy form - it explicitly specifies that the target distribution is uniform.

3. **Switch Transformer form**: $L_{aux} = N \cdot \sum_i f_i \cdot p_i$, where $f_i$ is the frequency at which expert i is chosen (its proportion in the batch) and $p_i$ is the average routing probability of expert i. This form does not look like KL, but when expanded it is equivalent to encouraging both $f_i$ and $p_i$ to be close to $1/N$.

Regardless of the form, the essence is the same probabilistic operation: constraining the shape of the router's output categorical distribution toward uniform. This is a typical engineering application of categorical distribution shape control.

**Why does the router use softmax rather than sigmoid**? Because top-k expert selection requires "each token picks k mutually exclusive experts", which is a truncated sample from a categorical distribution; sigmoid outputs "each expert independently chosen or not", a different semantics.


In [ ]:
# Demonstrate the categorical distribution output by the router using a tiny MoE
import torch
import torch.nn.functional as F

torch.manual_seed(42)

d_model = 8
num_experts = 4
top_k = 2

# Minimal router: a single linear layer
W_gate = torch.randn(d_model, num_experts)

# Simulate 3 tokens
tokens = torch.randn(3, d_model)
gate_logits = tokens @ W_gate  # [3, num_experts]
gate_probs = F.softmax(gate_logits, dim=-1)

print("=== Categorical distribution output by the router ===")
for i in range(3):
    print(f"token {i}: gate_logits = {[round(x, 3) for x in gate_logits[i].tolist()]}")
    print(f"         gate_probs   = {[round(x, 3) for x in gate_probs[i].tolist()]}")
    topk_vals, topk_idx = torch.topk(gate_logits[i], top_k)
    topk_weights = F.softmax(topk_vals, dim=-1)
    print(f"         top-{top_k} selection: experts {topk_idx.tolist()}, weights {[round(x, 3) for x in topk_weights.tolist()]}")
    print()

# Compute the batch-averaged routing distribution to demonstrate load balancing loss
mean_routing = gate_probs.mean(dim=0)  # [num_experts]
uniform = torch.ones(num_experts) / num_experts

# Form 1: negative entropy
neg_entropy = -(mean_routing * torch.log(mean_routing + 1e-12)).sum().item()

# Form 2: KL(mean_p || uniform)
kl_to_uniform = (mean_routing * (torch.log(mean_routing + 1e-12) - torch.log(uniform + 1e-12))).sum().item()

print("=== Two forms of the load balancing loss ===")
print(f"batch-averaged routing distribution: {[round(x, 4) for x in mean_routing.tolist()]}")
print(f"uniform distribution:                {[round(x, 4) for x in uniform.tolist()]}")
print(f"-H(avg distribution)       = {neg_entropy:.4f}  (smaller means closer to uniform)")
print(f"KL(avg || uniform)         = {kl_to_uniform:.4f}  (smaller means closer to uniform)")
print()
print("Key observation: both loss forms measure how far the average routing distribution deviates from uniform.")
print("Minimizing either of them encourages the router to distribute tokens evenly across all experts and prevents a few experts from being overloaded.")


## Summary

Confirm that you understand the following points:

- [ ] At each position an LLM outputs a categorical distribution, and softmax turns logits into valid probabilities
- [ ] A numerically stable softmax uses the logsumexp trick (subtract the max); this is the starting point of FlashAttention's online softmax
- [ ] Temperature divides the logits by T; T->0 is equivalent to greedy, T->inf is equivalent to uniform
- [ ] Top-k is truncation by a fixed count plus renormalization; top-p is truncation by cumulative probability plus renormalization
- [ ] Entropy measures the uncertainty of a distribution itself - maximal for uniform, zero for one-hot
- [ ] Cross-entropy measures "the cost of encoding p with q"; KL measures "the difference between two distributions"
- [ ] The core identity $H(p, q) = H(p) + D_{KL}(p \| q)$ ties the three together
- [ ] Under teacher forcing the true distribution is one-hot, $H(p) = 0$, so NLL = CE = KL
- [ ] PPL = exp(CE), meaning "the effective number of candidates per position"; it is not comparable across datasets
- [ ] The gradient form of softmax + CE is extremely clean: $\partial L / \partial z = p - \text{one\_hot}(t)$; its boundedness makes training stable
- [ ] The reuse map: training / RLHF / DPO / distillation / evaluation / inference / MoE all operate on the same set of mathematical objects
- [ ] The MoE router output is a categorical distribution, and the load balancing loss is a constraint on the shape of that distribution (entropy / KL form)

One-sentence summary: what repeatedly appears in LLM engineering is the categorical distribution and the three tools of information theory around it (entropy, cross-entropy, KL); once you grasp this set of mathematical objects, training, inference, alignment, and evaluation align on the same map.


## Exercises

> You may ask an AI to explain the approach, break down the steps, or check the direction, but it is not advisable to have the AI "just solve the problem for you".

**Exercise 1: Compute entropy by hand**

Given the distribution $p = [0.5, 0.25, 0.125, 0.125]$, compute $H(p)$ by hand.

Hint: $H(p) = -\sum p_i \log p_i$ using the natural logarithm.


In [ ]:
import math
import torch

p = torch.tensor([0.5, 0.25, 0.125, 0.125])

# TODO: compute H(p) by hand
manual_H = None  # fill in your hand-computed result here

# Verify with PyTorch
torch_H = -(p * torch.log(p)).sum().item()

assert manual_H is not None, "please compute manual_H first"
assert abs(manual_H - torch_H) < 1e-6, f"answer should be {torch_H:.6f}, you got {manual_H}"

print(f"✅ Exercise 1 passed:")
print(f"   H(p) = {manual_H:.4f}")
print(f"   theoretical maximum log(4) = {math.log(4):.4f}")
print(f"   p is more certain than uniform, so its entropy is smaller than log(4)")


**Exercise 2: Implement a numerically stable log-softmax**

Given a set of logits (including large numbers), implement a numerically stable log-softmax using the logsumexp trick. You may not call `F.log_softmax` directly.

Hint: $\log p_i = z_i - \text{logsumexp}(z)$, where $\text{logsumexp}(z) = c + \log \sum_j e^{z_j - c}$ and $c = \max_j z_j$.


In [ ]:
import torch
import torch.nn.functional as F

big_logits = torch.tensor([1000.0, 1001.0, 1002.0, 999.0])

def stable_log_softmax(z):
    """Numerically stable log-softmax without directly calling F.log_softmax"""
    # TODO: implement using the logsumexp trick
    c = z.max()
    logsumexp = None  # fill in c + log(sum(exp(z - c)))
    log_probs = None  # fill in z - logsumexp
    return log_probs

manual_lp = stable_log_softmax(big_logits)
torch_lp = F.log_softmax(big_logits, dim=-1)

assert manual_lp is not None, "please implement stable_log_softmax first"
assert torch.allclose(manual_lp, torch_lp, atol=1e-5), \
    f"result inconsistent with PyTorch\nyou got:    {manual_lp.tolist()}\nPyTorch: {torch_lp.tolist()}"

print("✅ Exercise 2 passed:")
print(f"   log_softmax = {[round(x, 4) for x in manual_lp.tolist()]}")
print(f"   corresponding probs = {[round(x, 4) for x in torch.exp(manual_lp).tolist()]}")
print(f"   sum of probs = {torch.exp(manual_lp).sum().item():.6f}")
print("   Key: subtract max before exp to avoid overflow on large numbers like 1000.")


**Exercise 3: Measure PPL in practice**

Given a set of logits and the corresponding targets, compute the average CE and PPL, and explain the meaning of PPL.

Hint: PPL = exp(average CE); use `F.cross_entropy` to compute CE and `math.exp` to compute PPL.


In [ ]:
import math
import torch
import torch.nn.functional as F

V = 8
logits = torch.tensor([
    [2.0, 1.0, 0.5, -0.5, 0.0, 0.2, -1.0, 0.1],
    [0.1, 0.2, 0.3, 0.1, 0.2, 0.3, 0.1, 0.2],  # very unsure
    [5.0, 0.1, 0.0, -1.0, 0.2, -0.5, 0.1, -0.3],  # very confident
])
targets = torch.tensor([0, 4, 0])

# TODO: compute average CE and PPL
ce = None       # use F.cross_entropy
ppl = None      # use math.exp(ce)

assert ce is not None and ppl is not None, "please compute ce and ppl first"

# Verify
expected_ce = F.cross_entropy(logits, targets).item()
expected_ppl = math.exp(expected_ce)
assert abs(ce - expected_ce) < 1e-6
assert abs(ppl - expected_ppl) < 1e-3

print(f"✅ Exercise 3 passed:")
print(f"   average CE = {ce:.4f}")
print(f"   PPL       = {ppl:.4f}")
print(f"   vocabulary V = {V}")
print()
print(f"   Interpretation:")
print(f"   PPL={ppl:.2f} means the model is effectively hesitating uniformly among about {ppl:.1f} candidates at each position")
print(f"   Position 2 (logits=5.0) is confident and contributes the least to CE; position 1 (nearly uniform) contributes the most")
print(f"   If PPL is close to V={V}, the model is essentially guessing; if it is close to 1, the model is almost entirely correct")
